In [ ]:
!pip install -q -U torch transformers peft datasets bitsandbytes trl accelerate

In [ ]:
!pip install evaluate sacremoses sacrebleu

In [ ]:
# 1. Clone the repository
!git clone https://github.com/vanboefer/lint_ii.git

# 2. Hack the config file: Change "requires-python <3.12" to "<3.14"
# This tells the installer that Python 3.12 is allowed.
!sed -i 's/<3.12/<3.14/' lint_ii/pyproject.toml

# 3. Install the modified package locally
!pip install ./lint_ii

# 4. Download the required Dutch model
!python -m spacy download nl_core_news_lg

In [ ]:
!pip install textstat bert_score

In [ ]:
from huggingface_hub import login

login()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "UWV/leesplank-noot-eurollm-1.7b"

# Quantization configuration for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "right"

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    use_cache=False
)

In [ ]:
from datasets import load_dataset
from peft import LoraConfig

#Load and format data
dataset = load_dataset("csv", data_files='/content/drive/MyDrive/complex-simple-v1-anonymized.csv')

# The model uses <|im_end|> to stop.
# tell the tokenizer that <|im_end|> is the eos_token.

#Set EOS to <|im_end|>
tokenizer.eos_token = "<|im_end|>"

# Set PAD to <|im_end|>
tokenizer.pad_token = "<|im_end|>"

# Enforce padding side
tokenizer.padding_side = "right"

print(f"EOS Token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
print(f"PAD Token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")

def format_instruction_safe(sample):
    messages = [
        {"role": "user", "content": f"Vereenvoudig: {sample['Complex']}"},
        {"role": "assistant", "content": sample['Simple']}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False)

    prompt = prompt.strip()

    return {"text": prompt}

dataset = dataset.map(format_instruction_safe)


split_1 = dataset["train"].train_test_split(test_size=160, seed=42)
test_dataset = split_1["test"]
remaining_dataset = split_1["train"]


split_2 = remaining_dataset.train_test_split(test_size=150, seed=42)
eval_dataset = split_2["test"]
train_dataset = split_2["train"]

print(f"Train: {len(train_dataset)}")
print(f"Eval:  {len(eval_dataset)}")
print(f"Test:  {len(test_dataset)}")


peft_config = LoraConfig(
    task_type="CAUSAL_LM",
    bias="none",
    # r=8,
    # lora_alpha=8,
    # lora_dropout=0.0,
    # target_modules

print(peft_config.to_dict())


In [ ]:
train_dataset[1]



In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    # learning_rate=7e-5,
    # weight_decay=0.01,
    # lr_scheduler_type="cosine",
    # warmup_steps=100,
    # max_grad_norm=1.0,

    # EVAL CONFIGURATION
    eval_strategy="steps",           # Evaluate during training
    eval_steps=50,                   # Every 50 training steps
    save_strategy="steps",           # Save checkpoints
    save_steps=50,                   # Every 50 steps
    save_total_limit=3,              # Keep only best 3 checkpoints
    load_best_model_at_end=True,    # Load best checkpoint after training
    metric_for_best_model="eval_loss",  # Use eval loss to pick best
    greater_is_better=False,         # Lower eval loss is better

    logging_steps=10,
    # bf16=True,
    # optim="paged_adamw_32bit",
    # report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    args=sft_config,
)

# Training config:
print("Training with these configs:")
print(sft_config.to_dict())
# Train and automatically evaluate
# trainer.train()

In [ ]:
# Save the LoRA adapter
#new_model_name = "leesplank-municipal-lora"
#trainer.model.save_pretrained(new_model_name)
#tokenizer.save_pretrained(new_model_name)


In [ ]:
#hub_model_id = ""

# Push the adapter and tokenizer
#trainer.model.push_to_hub(hub_model_id)
#tokenizer.push_to_hub(hub_model_id)

#print(f"Adapter uploaded to: https://huggingface.co/{hub_model_id}")

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Clear memory from training
#del model, trainer
#torch.cuda.empty_cache()


# Reload tokenizer
#base_model_id = "UWV/leesplank-noot-eurollm-1.7b"
#tokenizer = AutoTokenizer.from_pretrained(base_model_id)
#tokenizer.padding_side = "right"

# Reload base model in BF16
#base_model = AutoModelForCausalLM.from_pretrained(
#    base_model_id,
#    torch_dtype=torch.bfloat16,
#    device_map="auto"
#)



In [ ]:
# Load  LoRA adapter
#adapter_path = "./leesplank-municipal-lora"

#model_with_adapter = PeftModel.from_pretrained(base_model, adapter_path)

print("Adapter loaded")

# Merge adapter into base model
print("Merging adapter into base model...")
#merged_model = model_with_adapter.merge_and_unload()

print("Merge completed")
#print(f"Merged model size: {merged_model.get_memory_footprint() / 1024**3:.2f} GB")

# Verify model works
#test_text = "De gemeenteraad heeft een besluit genomen."
#messages = [{"role": "user", "content": f"Vereenvoudig: {test_text}"}]
#inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to(merged_model.device)
#outputs = merged_model.generate(inputs, max_new_tokens=50, do_sample=False)
#print("\Test generation:")
#print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
from huggingface_hub import notebook_login

# notebook_login()  # Run if not already logged in

#full_model_repo = ""

# Push merged model and tokenizer

#merged_model.push_to_hub(
 #   full_model_repo,
  #  commit_message="Upload merged model with municipal Dutch fine-tuning"
#)
#tokenizer.push_to_hub(full_model_repo)


In [ ]:
import evaluate
import textstat
import numpy as np
from bert_score import score as bert_score_func
from lint_ii import ReadabilityAnalysis
from transformers import AutoModelForCausalLM
from tqdm import tqdm
import logging

# Setup
sari = evaluate.load("sari")
logging.getLogger("lint_ii").setLevel(logging.ERROR)

# Prepare Data
sources = eval_dataset['Complex']
# References for SARI (list of lists)
references_sari = [[x] for x in eval_dataset['Simple']]
# References for BERT (flat list)
references_flat = list(eval_dataset['Simple'])
references = eval_dataset['Simple'] # For Flesch and Lint-II

# GENERATION FUNCTIONS

def generate_predictions(model, tokenizer, texts):
    predictions = []
    model.eval()

    if texts and hasattr(model, "device") and model.device.type == 'cpu':
        print("Model is running on CPU")

    stop_words = ["<|im_start|>", "system", "user", "Vereenvoudig:", "\n\n"]

    for text in tqdm(texts, desc="Generating"):
        messages = [{"role": "user", "content": f"Vereenvoudig: {text}"}]

        # Get Input IDs
        input_ids = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True
        ).to(model.device)

        # Create Attention Mask
        # Since batch size is 1, we attend to everything
        attention_mask = torch.ones_like(input_ids)

        with torch.no_grad():
            outputs = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")]
            )

        full_output = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)

        cleaned_output = full_output
        for stopper in stop_words:
            if stopper in cleaned_output:
                cleaned_output = cleaned_output.split(stopper)[0]

        cleaned_output = cleaned_output.strip()
        if not cleaned_output:
            # If model returns empty, use original text
            cleaned_output = text

        predictions.append(cleaned_output)

    return predictions

# RUN GENERATION
#

# Fine-Tuned Model
finetuned_model_id = ""
print(f"Loading Fine-Tuned Model: {finetuned_model_id}")
ft_model = AutoModelForCausalLM.from_pretrained(finetuned_model_id, torch_dtype=torch.bfloat16, device_map="auto")
ft_tokenizer = AutoTokenizer.from_pretrained(finetuned_model_id)

# Ensure pad token is set
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token


print("Generating Fine-Tuned Predictions")
ft_predictions = generate_predictions(ft_model, ft_tokenizer, sources)

# Free memory
del ft_model
torch.cuda.empty_cache()

# Base Model
base_model_id = ""
print(f"Loading Base Model: {base_model_id}")
base_model = AutoModelForCausalLM.from_pretrained(base_model_id, torch_dtype=torch.bfloat16, device_map="auto")
base_tokenizer = AutoTokenizer.from_pretrained(base_model_id)

print("Generating Base Model Predictions")
base_predictions = generate_predictions(base_model, base_tokenizer, sources)

# Free memory
del base_model
torch.cuda.empty_cache()

In [ ]:

# CALCULATE ALL METRICS

sources = list(eval_dataset['Complex'])
references_flat = list(eval_dataset['Simple'])
references_sari = [[x] for x in references_flat]
references = references_flat

# SARI
print("Calculating SARI...")
sari_id = sari.compute(sources=sources, predictions=sources, references=references_sari)
sari_ref = sari.compute(sources=sources, predictions=references, references=references_sari)
sari_base = sari.compute(sources=sources, predictions=base_predictions, references=references_sari)
sari_ft = sari.compute(sources=sources, predictions=ft_predictions, references=references_sari)

# FLESCH-DOUMA
print("Calculating Flesch-Douma")
textstat.set_lang("nl_NL")

def get_flesch_douma(texts):
    scores = []
    for t in texts:
        if t and len(t.strip()) > 5:
            try:
                scores.append(textstat.flesch_reading_ease(t))
            except:
                continue
    return np.mean(scores) if scores else 0.0

fd_src = get_flesch_douma(sources)
fd_simp = get_flesch_douma(references)
fd_base = get_flesch_douma(base_predictions)
fd_ft = get_flesch_douma(ft_predictions)

# BERTSCORE
print("Calculating BERTScore")
def get_bert_src(preds):
    P, R, F1 = bert_score_func(preds, sources, lang="nl", verbose=False, device='cuda' if torch.cuda.is_available() else 'cpu')
    return F1.mean().item()
sources_flat = list(sources)
bert_sources_src = get_bert_src(sources_flat)
bert_references_src = get_bert_src(references_flat)
bert_base_src = get_bert_src(base_predictions)
bert_ft_src = get_bert_src(ft_predictions)


# LiNT-II
print("Calculating LiNT-II (Dutch Gov Readability)")
def get_lint(texts, name):
    scores, levels = [], []
    print(f"Analyzing {name}...")
    for text in tqdm(texts):
        if not text or len(text.strip()) < 5: continue
        try:
            analysis = ReadabilityAnalysis.from_text(text)
            if analysis.lint.score is not None and analysis.lint.level is not None:
                scores.append(analysis.lint.score)
                levels.append(analysis.lint.level)
        except Exception:
            continue

    if not scores:
        print(f"LiNT-II calculation failed for all texts in '{name}'. Returning 0.0.")
        return 0.0, 0.0

    return np.mean(scores), np.mean(levels)

lint_score_src, lint_level_src = get_lint(sources, "Source Text")
lint_score_ref, lint_level_ref = get_lint(references, "References Text")
lint_score_base, lint_level_base = get_lint(base_predictions, "Base UWV Model")
lint_score_ft, lint_level_ft = get_lint(ft_predictions, "Fine-Tuned Model")

print("All calculations complete.")

# SUMMARY TABLE
print("\n" + "="*50)
print(f"{'METRIC':<20} | {'SOURCE/ID':<12} | {'REFERENCE':<12} |{'BASE MODEL':<12} | {'FINE-TUNED':<12}")
print("-" * 50)
print(f"{'SARI (Higher better)':<20} | {sari_id['sari']:<12.2f} | {sari_ref['sari']:<12.2f} | {sari_base['sari']:<12.2f} | {sari_ft['sari']:<12.2f}")
print(f"{'Flesch-Douma (H)':<20} | {fd_src:<12.2f} |{fd_simp:<12.2f} |{fd_base:<12.2f} | {fd_ft:<12.2f}")
print(f"{'LiNT-II Score (L)':<20} | {lint_score_src:<12.2f} | {lint_score_ref:<12.2f} | {lint_score_base:<12.2f} | {lint_score_ft:<12.2f}")
print(f"{'LiNT-II Level (L)':<20} | {lint_level_src:<12.2f} | {lint_level_ref:<12.2f} | {lint_level_base:<12.2f} | {lint_level_ft:<12.2f}")
print(f"{'BERTScore Meaning':<20} | {bert_sources_src:<12.4f} | {bert_references_src:<12.2f} | {bert_base_src:<12.4f} | {bert_ft_src:<12.4f}")



print("="*50)

In [ ]:
# LENGTHS

import re
import numpy as np

#
# CALCULATE AVERAGE WORD LENGTH

def get_avg_word_length(texts):
    total_len = 0
    total_count = 0

    for text in texts:
        if isinstance(text, str) and len(text) > 0:
            # Use regex to find alphanumeric words (excludes punctuation)
            words = re.findall(r'\w+', text)
            if words:
                total_len += sum(len(w) for w in words)
                total_count += len(words)

    return total_len / total_count if total_count > 0 else 0.0

# Calculate metrics
wl_src = get_avg_word_length(sources)
wl_ref = get_avg_word_length(references)
wl_base = get_avg_word_length(base_predictions)
wl_ft = get_avg_word_length(ft_predictions)

# Print results
print(f"{'Source (Complex)':<25}: {wl_src:.2f} chars/word")
print(f"{'Reference (Simple)':<25}: {wl_ref:.2f} chars/word")
print(f"{'Base Model':<25}: {wl_base:.2f} chars/word")
print(f"{'Fine-Tuned':<25}: {wl_ft:.2f} chars/word")

print("\n(Comparison row for your table:)")
print(f"{'Avg Word Len (L)':<20} | {wl_src:<12.2f} | {wl_ref:<12.2f} | {wl_base:<12.2f} | {wl_ft:<12.2f}")


In [ ]:
import re
import numpy as np

# CALCULATE AVERAGE SENTENCE LENGTH (WORDS PER SENTENCE)
def get_avg_sentence_length(texts):
    word_counts = []
    for text in texts:
        if isinstance(text, str) and len(text.strip()) > 0:
            # re.findall(r'\w+', text) finds all alphanumeric words
            # This ignores punctuation marks as "words"
            words = re.findall(r'\w+', text)
            word_counts.append(len(words))

    return np.mean(word_counts) if word_counts else 0.0

# Calculate metrics
sl_src = get_avg_sentence_length(sources)
sl_ref = get_avg_sentence_length(references)
sl_base = get_avg_sentence_length(base_predictions)
sl_ft = get_avg_sentence_length(ft_predictions)

# Print results
print(f"{'Source (Complex)':<25}: {sl_src:.2f} words/sent")
print(f"{'Reference (Simple)':<25}: {sl_ref:.2f} words/sent")
print(f"{'Base Model':<25}: {sl_base:.2f} words/sent")
print(f"{'Fine-Tuned':<25}: {sl_ft:.2f} words/sent")

# Table row formatted for your previous summary style
print("\n(Comparison row for your table:)")
print(f"{'Words/Sent (L)':<20} | {sl_src:<12.2f} | {sl_ref:<12.2f} | {sl_base:<12.2f} | {sl_ft:<12.2f}")


In [ ]:
import json
import os
from datetime import datetime


MANUAL_NOTE = "Iteration 1: Baseline vs Fine-Tuned (Full Metrics including References)"

experiment_report = {
    "meta": {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "base_model": "",
        "dataset_split": "1000 Train / 150 Val / 160 Test",
        "notes": MANUAL_NOTE
    },

    "configs": {
        "peft_lora_config": peft_config.to_dict(),
        "sft_training_args": sft_config.to_dict()
    },

    "results": {
        "sari": {
            "identity_baseline": sari_id['sari'],
            "human_reference_ceiling": sari_ref['sari'],
            "base_model": sari_base['sari'],
            "finetuned": sari_ft['sari']
        },
        "lint_ii": {
            "source": {"score": lint_score_src, "level": lint_level_src},
            "reference": {"score": lint_score_ref, "level": lint_level_ref},
            "base_model": {"score": lint_score_base, "level": lint_level_base},
            "finetuned": {"score": lint_score_ft, "level": lint_level_ft}
        },
        "flesch_douma": {
            "source": fd_src,
            "reference": fd_simp,
            "base_model": fd_base,
            "finetuned": fd_ft
        },
        "bert_score_meaning_preservation": {
            "description": "F1 score comparing Output against Complex Source (1.0 = exact preservation)",
            "source_identity": bert_sources_src,     # Should be 1.0
            "human_reference": bert_references_src,  # Methodology Target
            "base_model": bert_base_src,
            "finetuned": bert_ft_src
        }
    }
}

# SAVE TO GOOGLE DRIVE

filename = f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_experiment_results_FULL.json"
save_path = f"/content/drive/MyDrive/{filename}"

try:
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(experiment_report, f, indent=4, default=str)
    print(f"Report successfully saved to: {save_path}")

except Exception as e:
    print(f"Drive Error ({e}). Saving locally.")
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(experiment_report, f, indent=4, default=str)
    print(f"Saved locally at: {filename}")



In [ ]:
# Print 5 examples to see what is actually happening

sources = eval_dataset['Complex']

references = eval_dataset['Simple']
for i in range(5):
    src = sources[i]
    ref = references[i]
    base_pred = base_predictions[i]
    ft_pred = ft_predictions[i]

    print(f"\nExample {i+1}:")
    print(f"Source:   {src}")
    print(f"Reference: {ref}")
    print(f"Base:      {base_pred}")
    print(f"Fine-Tune: {ft_pred}")
    print("-" * 30)

In [ ]:
# Check for empty predictions
empty_base = [i for i, pred in enumerate(base_predictions) if not pred.strip()]
empty_ft = [i for i, pred in enumerate(ft_predictions) if not pred.strip()]

print(f"Base model empty outputs: {len(empty_base)}")
print(f"Fine-tuned empty outputs: {len(empty_ft)}")

if empty_ft:
    print("\nExamples of inputs that produced empty outputs:")
    for idx in empty_ft[:3]:
        print(f"Input {idx}: {sources[idx]}")

Data exploration


In [ ]:

#EXPLORATORY DATA ANALYSIS - TRAINING SET ONLY
# We only analyze the training set to avoid data leakage
# from validation/test sets influencing our understanding.

import pandas as pd
import numpy as np

print("DATASET ANALYSIS - TRAINING SET (N={})".format(len(train_dataset)))

# IDENTITY SIMPLIFICATIONS (Source = Reference)
train_sources = train_dataset['Complex']
train_references = train_dataset['Simple']

# Check exact matches
identical_pairs = [
    (src, ref) for src, ref in zip(train_sources, train_references)
    if src.strip() == ref.strip()
]

num_identical = len(identical_pairs)
pct_identical = (num_identical / len(train_dataset)) * 100

print(f"IDENTITY SIMPLIFICATIONS")
print(f"Pairs where Source = Reference: {num_identical} / {len(train_dataset)}")
print(f"Percentage: {pct_identical:.1f}%")

# SIMILARITY ANALYSIS
# Check near-identical pairs (only minor punctuation/whitespace differences)
near_identical = 0
for src, ref in zip(train_sources, train_references):
    src_normalized = src.strip().lower().replace('.', '').replace(',', '')
    ref_normalized = ref.strip().lower().replace('.', '').replace(',', '')
    if src_normalized == ref_normalized and src != ref:
        near_identical += 1

print(f"NEAR-IDENTICAL PAIRS (only punctuation differs)")
print(f"Count: {near_identical}")
print(f"Minimal simplification applied")

# TEXT LENGTH STATISTICS
src_lengths = [len(text.split()) for text in train_sources]
ref_lengths = [len(text.split()) for text in train_references]

print(f"\TEXT LENGTH STATISTICS (word count)")
print(f"{'Metric':<20} | {'Source':<12} | {'Reference':<12} | {'Δ':<12}")
print(f"{'-'*60}")
print(f"{'Mean':<20} | {np.mean(src_lengths):<12.1f} | {np.mean(ref_lengths):<12.1f} | {np.mean(ref_lengths) - np.mean(src_lengths):<12.1f}")
print(f"{'Median':<20} | {np.median(src_lengths):<12.1f} | {np.median(ref_lengths):<12.1f} | {np.median(ref_lengths) - np.median(src_lengths):<12.1f}")
print(f"{'Min':<20} | {np.min(src_lengths):<12.0f} | {np.min(ref_lengths):<12.0f} | {np.min(ref_lengths) - np.min(src_lengths):<12.0f}")
print(f"{'Max':<20} | {np.max(src_lengths):<12.0f} | {np.max(ref_lengths):<12.0f} | {np.max(ref_lengths) - np.max(src_lengths):<12.0f}")

# SIMPLIFICATION OPERATIONS
shortened = sum(1 for s, r in zip(src_lengths, ref_lengths) if r < s)
lengthened = sum(1 for s, r in zip(src_lengths, ref_lengths) if r > s)
same_length = sum(1 for s, r in zip(src_lengths, ref_lengths) if r == s)

print(f"SIMPLIFICATION OPERATIONS")
print(f"Shortened (ref < source):  {shortened} ({shortened/len(train_dataset)*100:.1f}%)")
print(f"Lengthened (ref > source): {lengthened} ({lengthened/len(train_dataset)*100:.1f}%)")
print(f"Same length:               {same_length} ({same_length/len(train_dataset)*100:.1f}%)")

# EXAMPLES OF IDENTITY SIMPLIFICATIONS
if num_identical > 0:
    print(f"EXAMPLES OF IDENTITY SIMPLIFICATIONS")
    print(f"Showing first 5 cases where Source = Reference")
    print()
    for i, (src, ref) in enumerate(identical_pairs[:5], 1):
        print(f"Example {i}:")
        print(f"{src}")
        print()

